In [3]:
import plotly.express as px
import numpy as np
from ase.io import read
import os

In [ ]:
# SO3_NMe4

from ase.io import read
from ase.neighborlist import NeighborList
import math

def mean_counterionN_to_cluster_distance(filename, cutoff=1.6):
    """
    Mean minimum distance between N centers of N(CH3)4+ counterions
    and the cluster atoms.
    """

    atoms = read(filename)
    symbols = atoms.get_chemical_symbols()
    positions = atoms.get_positions()

    # Build neighbor list
    cutoffs = [cutoff] * len(atoms)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)

    counterion_N_indices = []
    counterion_atom_indices = set()

    # Identify N(CH3)4+ nitrogens
    for i, s in enumerate(symbols):
        if s != 'N':
            continue

        neighbors, _ = nl.get_neighbors(i)
        neighbor_symbols = [symbols[j] for j in neighbors]

        # N bonded to exactly four carbons → N(CH3)4+
        if neighbor_symbols.count('C') == 4:
            counterion_N_indices.append(i)
            counterion_atom_indices.add(i)
            counterion_atom_indices.update(neighbors)

    if not counterion_N_indices:
        raise ValueError("No N(CH3)4+ counterions found.")

    # Cluster atoms = everything not in counterions
    cluster_positions = [
        positions[i]
        for i in range(len(atoms))
        if i not in counterion_atom_indices
    ]

    min_distances = []

    for i in counterion_N_indices:
        Nx, Ny, Nz = positions[i]

        dmin = float('inf')
        for x, y, z in cluster_positions:
            d = math.sqrt((x - Nx)**2 + (y - Ny)**2 + (z - Nz)**2)
            if d < dmin:
                dmin = d

        min_distances.append(dmin)

    return sum(min_distances) / len(min_distances)

dist_list = []
energy_to_distance = {}

for idx in range(1, 2001):
    filename = f'./SO3_NMe4/results_xtb/{idx}/xtbopt.xyz'
    if os.path.exists(filename):
        # Read energy from second line
        with open(filename, 'r') as f:
            lines = f.readlines()
            second_line = lines[1].strip()
            energy_str = second_line.split()[1]  # assumes "energy: <value>"
            energy = float(energy_str)

        # Compute mean N⁺–cluster distance
        mean_dist = mean_counterionN_to_cluster_distance(filename) - 1.52
        dist_list.append(mean_dist)
        energy_to_distance[energy] = mean_dist



In [7]:

num_bins = 20
min_d, max_d = min(dist_list), max(dist_list)
bin_width = (max_d - min_d) / num_bins
bins = [min_d + i*bin_width for i in range(num_bins+1)]
counts = [0]*num_bins

for d in dist_list:
    for i in range(num_bins):
        if bins[i] <= d < bins[i+1]:
            counts[i] += 1
            break
    else:
        if d == bins[-1]:
            counts[-1] += 1

# Normalize counts to %
max_count = max(counts)
counts_normalized = [c/max_count*100 for c in counts]
bin_centers = [(bins[i] + bins[i+1])/2 for i in range(num_bins)]

# --- Plot with Plotly ---
fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Mean NMe4-cage distance/ Å", "y": "Relative Counts %"},
    title="Distribution of Counterion Distances",
    color_discrete_sequence=['purple']
)

fig.update_xaxes(ticks="outside")
fig.update_yaxes(ticks="outside")
fig.update_layout(
    bargap=0.05,
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, zeroline=False, linecolor="black", mirror=True),
    yaxis=dict(showgrid=True, zeroline=False, linecolor="black", mirror=True)
)

fig.to_html('Distribution_counterions_So3_NMe4.html')
fig.show()

In [9]:
x = list(energy_to_distance.values())  # mean NMe4–cluster distances
y = [(energy - (-67.84989753) - (-2.40690122)*4)* 2625.5 for energy in energy_to_distance.keys()]

fig = px.scatter(
    x=x,
    y=y,
    labels={"x": "Mean NMe4-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Energy vs. Mean NMe4-cage distance",
    width=700,
    height=500
)
fig.update_traces(marker=dict(size=10, color='purple', line=dict(width=1, color='black')))  # size=10 for larger points

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()

In [ ]:
reference_energy = (-67.84989753) + (-2.40690122)*4

distances_all = list(energy_to_distance.values())
energies_all = [(energy - reference_energy) * 2625.5 for energy in energy_to_distance.keys()]

distances = [d for d, e in zip(distances_all, energies_all) if e < 0]
energies = [e for e in energies_all if e < 0]

fig = px.scatter(
    x=distances,
    y=energies,
    labels={"x": "Mean NMe4-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Cohesive energy vs. Mean NMe4-cage distance (E < 0)",
    marginal_x="histogram",
    marginal_y="histogram",
    width=800,
    height=600
)

fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=10, color='purple', line=dict(width=1, color='black'))
)
fig.update_traces(
    selector=dict(type="histogram"),
    marker=dict(color="purple"),
    opacity=0.5
)
for trace in fig.data:
    if trace.type == "histogram":
        trace.nbinsx = 30  # for top histogram (x axis)
        trace.nbinsy = 50  # for right histogram (y axis)
fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()

In [31]:
x = list(energy_to_distance.values())  # mean PF6–cluster distances
y = [(energy - (-63.32357097) - (-0.657269107)*8)* 2625.5 for energy in energy_to_distance.keys()]

fig = px.scatter(
    x=x,
    y=y,
    labels={"x": "Mean PF6-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Energy vs. Mean PF6-cage distance",
    width=700,
    height=500
)
fig.update_traces(marker=dict(size=10, color='hotpink', line=dict(width=1, color='black')))  # size=10 for larger points

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()

In [32]:
y = [(energy - (-63.32357097) - (-0.657269107)*8)* 2625.5 for energy in energy_to_distance.keys()]
y_negative = [val for val in y if val < 0]
counts, bins = np.histogram(y_negative, bins=20)
counts_normalized = counts / counts.max() * 100
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Cohesive energy/ kJ mol\u207B\u00B9", "y": "Relative Counts %"},
    title="Histogram of Relative Energies (Max = 100)",
    color_discrete_sequence=['hotpink']
)

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    bargap=0.05
)

fig.show()

In [33]:
reference_energy = -63.32357097 + (-0.657269107)*8

distances_all = list(energy_to_distance.values())
energies_all = [(energy - reference_energy) * 2625.5 for energy in energy_to_distance.keys()]

distances = [d for d, e in zip(distances_all, energies_all) if e < 0]
energies = [e for e in energies_all if e < 0]

fig = px.scatter(
    x=distances,
    y=energies,
    labels={"x": "Mean PF6–cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Cohesive energy vs. Mean PF6–cage Distance (E < 0)",
    marginal_x="histogram",
    marginal_y="histogram",
    width=800,
    height=600
)

fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=10, color='hotpink', line=dict(width=1, color='black'))
)
fig.update_traces(
    selector=dict(type="histogram"),
    marker=dict(color="hotpink"),
    opacity=0.7
)
for trace in fig.data:
    if trace.type == "histogram":
        trace.nbinsx = 30  # for top histogram (x axis)
        trace.nbinsy = 30  # for right histogram (y axis)
fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()



In [60]:
# OH_SO4

def mean_counterion_to_cluster_distance(filename):

    atoms = read(filename)
    symbols = np.array(atoms.get_chemical_symbols())
    positions = atoms.get_positions()

    # Indices of PF6 phosphorus atoms
    P_indices = np.where(symbols == 'S')[0]

    if len(P_indices) == 0:
        raise ValueError("No phosphorus atoms found.")

    # Atoms to consider as cluster surface (exclude F and P)
    target_mask = (symbols != 'O') & (symbols != 'S')
    target_positions = positions[target_mask]

    min_distances = []

    for i in P_indices:
        P_pos = positions[i]

        dists = np.linalg.norm(target_positions - P_pos, axis=1)
        min_distances.append(dists.min())

    return float(np.mean(min_distances))

dist_list = []
energy_to_distance = {}
for idx in range(1, 2001):
    filename = f'./xtb_OH_SO4/{idx}/xtbopt.xyz'
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            lines = f.readlines()
            second_line = lines[1].strip()
            energy_str = second_line.split()[1]  # second element after "energy:"
            energy = float(energy_str)
        mean_dist = mean_counterion_to_cluster_distance(filename) - 1.44
        dist_list.append(mean_dist)
        energy_to_distance[energy] = mean_dist

counts, bins = np.histogram(dist_list, bins=20)
counts_normalized = counts / counts.max() * 100
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Mean SO4-cage surface distance/ Å", "y": "Relative Counts %"},
    title="OH_SO4"
)
fig.update_xaxes(
    ticks="outside",
    #ticklen=8,
    #tickwidth=2,
    tickcolor="black"
)

fig.update_yaxes(
    ticks="outside",
    #ticklen=8,
    #tickwidth=2,
    tickcolor="black"
)
fig.update_layout(
    bargap=0.05,
    plot_bgcolor="white",        # white background
    xaxis=dict(
        showgrid=True,           # subtle grid
        #gridcolor="lightgray",
        zeroline=False,
        linecolor="black",
        mirror=True              # borders on top/bottom
    ),
    yaxis=dict(
        showgrid=True,
        #gridcolor="lightgray",
        zeroline=False,
        linecolor="black",
        mirror=True              # borders left/right
    ),
)
fig.to_html('Ditribution_counterions_OH_SO4.html')
fig.show()

In [61]:
x = list(energy_to_distance.values())  # mean PF6–cluster distances
y = [(energy - (-63.32357097) - (-0.703061194)*4)* 2625.5 for energy in energy_to_distance.keys()]

fig = px.scatter(
    x=x,
    y=y,
    labels={"x": "Mean SO4-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Energy vs. Mean SO4-cage distance",
    width=700,
    height=500
)
fig.update_traces(marker=dict(size=10, color='blue', line=dict(width=1, color='black')))  # size=10 for larger points

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()

In [38]:
y = [(energy - (-63.32357097) - (-0.703061194)*4)* 2625.5 for energy in energy_to_distance.keys()]
y_negative = [val for val in y if val < 0]
counts, bins = np.histogram(y_negative, bins=20)
counts_normalized = counts / counts.max() * 100
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Cohesive energy/ kJ mol\u207B\u00B9", "y": "Relative Counts %"},
    title="Histogram of Relative Energies (Max = 100)"
)

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    bargap=0.05
)

fig.show()

In [62]:
reference_energy = -63.32357097 + (-0.703061194)*4

distances_all = list(energy_to_distance.values())
energies_all = [(energy - reference_energy) * 2625.5 for energy in energy_to_distance.keys()]

distances = [d for d, e in zip(distances_all, energies_all) if e < 0]
energies = [e for e in energies_all if e < 0]

fig = px.scatter(
    x=distances,
    y=energies,
    labels={"x": "Mean SO4-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Relative Energy vs. Mean SO4- Cluster Distance (E < 0)",
    marginal_x="histogram",
    marginal_y="histogram",
    width=800,
    height=600
)

fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=10, color='blue', line=dict(width=1, color='black'))
)
for trace in fig.data:
    if trace.type == "histogram":
        trace.nbinsx = 30  # for top histogram (x axis)
        trace.nbinsy = 30  # for right histogram (y axis)
fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()


In [63]:
# SO3_Na
def mean_counterion_to_cluster_distance(filename):

    atoms = read(filename)
    symbols = np.array(atoms.get_chemical_symbols())
    positions = atoms.get_positions()

    # Indices of PF6 phosphorus atoms
    P_indices = np.where(symbols == 'Na')[0]

    if len(P_indices) == 0:
        raise ValueError("No phosphorus atoms found.")

    # Atoms to consider as cluster surface (exclude F and P)
    target_mask = (symbols != 'Na')
    target_positions = positions[target_mask]

    min_distances = []

    for i in P_indices:
        P_pos = positions[i]

        dists = np.linalg.norm(target_positions - P_pos, axis=1)
        min_distances.append(dists.min())

    return float(np.mean(min_distances))

dist_list = []
energy_to_distance = {}
for idx in range(1, 2001):
    filename = f'./SO3_Na_new/{idx}/xtbopt.xyz'
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            lines = f.readlines()
            second_line = lines[1].strip()
            energy_str = second_line.split()[1]  # second element after "energy:"
            energy = float(energy_str)
        mean_dist = mean_counterion_to_cluster_distance(filename)
        dist_list.append(mean_dist)
        energy_to_distance[energy] = mean_dist

counts, bins = np.histogram(dist_list, bins=20)
counts_normalized = counts / counts.max() * 100
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Mean Na-cage surface distance/ Å", "y": "Relative Counts %"},
    title="SO3_Na",
    color_discrete_sequence=['red']
)
fig.update_xaxes(
    ticks="outside",
    #ticklen=8,
    #tickwidth=2,
    tickcolor="black"
)

fig.update_yaxes(
    ticks="outside",
    #ticklen=8,
    #tickwidth=2,
    tickcolor="black"
)
fig.update_layout(
    bargap=0.05,
    plot_bgcolor="white",        # white background
    xaxis=dict(
        showgrid=True,           # subtle grid
        #gridcolor="lightgray",
        zeroline=False,
        linecolor="black",
        mirror=True              # borders on top/bottom
    ),
    yaxis=dict(
        showgrid=True,
        #gridcolor="lightgray",
        zeroline=False,
        linecolor="black",
        mirror=True              # borders left/right
    ),
)
fig.to_html('Ditribution_counterions_SO3_Na.html')
fig.show()


In [64]:
x = list(energy_to_distance.values())  # mean PF6–cluster distances
y = [(energy - (-67.849897530023))* 2625.5 for energy in energy_to_distance.keys()]

fig = px.scatter(
    x=x,
    y=y,
    labels={"x": "Mean Na-cage Distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Energy vs. Mean Na-cage Distance",
    width=700,
    height=500
)
fig.update_traces(marker=dict(size=10, color='red', line=dict(width=1, color='black')))  # size=10 for larger points

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()

In [65]:
y = [(energy - (-67.849897530023))* 2625.5 for energy in energy_to_distance.keys()]
y_negative = [val for val in y if val < 0]
counts, bins = np.histogram(y_negative, bins=20)
counts_normalized = counts / counts.max() * 100
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Cohesive energy/ kJ mol\u207B\u00B9", "y": "Relative Counts %"},
    title="Histogram of Relative Energies (Max = 100)",
    color_discrete_sequence=['red']
)

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    bargap=0.05
)

fig.show()

In [66]:
reference_energy = (-67.849897530023)

distances_all = list(energy_to_distance.values())
energies_all = [(energy - reference_energy) * 2625.5 for energy in energy_to_distance.keys()]

distances = [d for d, e in zip(distances_all, energies_all) if e < 0]
energies = [e for e in energies_all if e < 0]

fig = px.scatter(
    x=distances,
    y=energies,
    labels={"x": "Mean Na-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Relative Energy vs. Mean Na-cage distance (E < 0)",
    marginal_x="histogram",
    marginal_y="histogram",
    width=800,
    height=600
)

fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=10, color='red', line=dict(width=1, color='black'))
)


# Marginal histograms
fig.update_traces(
    selector=dict(type="histogram"),
    marker=dict(color="red"),
    opacity=0.5
)
for trace in fig.data:
    if trace.type == "histogram":
        trace.nbinsx = 30  # for top histogram (x axis)
        trace.nbinsy = 30  # for right histogram (y axis)
fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()


In [50]:
# Me_SO4

def mean_counterion_to_cluster_distance(filename):
    """
    Mean of the minimum distances between each PF6 phosphorus atom
    and any atom that is not F or P.
    """

    atoms = read(filename)
    symbols = np.array(atoms.get_chemical_symbols())
    positions = atoms.get_positions()

    # Indices of PF6 phosphorus atoms
    P_indices = np.where(symbols == 'S')[0]

    if len(P_indices) == 0:
        raise ValueError("No phosphorus atoms found.")

    # Atoms to consider as cluster surface (exclude F and P)
    target_mask = (symbols != 'S') & (symbols != 'O')
    target_positions = positions[target_mask]

    min_distances = []

    for i in P_indices:
        P_pos = positions[i]

        dists = np.linalg.norm(target_positions - P_pos, axis=1)
        min_distances.append(dists.min())

    return float(np.mean(min_distances))

dist_list = []
energy_to_distance = {}
for idx in range(1, 2001):
    filename = f'./Me_SO4_new/{idx}/xtbopt.xyz'
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            lines = f.readlines()
            second_line = lines[1].strip()
            energy_str = second_line.split()[1]  # second element after "energy:"
            energy = float(energy_str)
        mean_dist = mean_counterion_to_cluster_distance(filename) -1.44
        dist_list.append(mean_dist)
        energy_to_distance[energy] = mean_dist

counts, bins = np.histogram(dist_list, bins=20)
counts_normalized = counts / counts.max() * 100
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Mean SO4-cage distance/ Å", "y": "Relative Counts %"},
    title="Me_So4",
    color_discrete_sequence=['green']
    
)
fig.update_xaxes(
    ticks="outside",
    #ticklen=8,
    #tickwidth=2,
    tickcolor="black"
)

fig.update_yaxes(
    ticks="outside",
    #ticklen=8,
    #tickwidth=2,
    tickcolor="black"
)
fig.update_layout(
    bargap=0.05,
    plot_bgcolor="white",        # white background
    xaxis=dict(
        showgrid=True,           # subtle grid
        #gridcolor="lightgray",
        zeroline=False,
        linecolor="black",
        mirror=True              # borders on top/bottom
    ),
    yaxis=dict(
        showgrid=True,
        #gridcolor="lightgray",
        zeroline=False,
        linecolor="black",
        mirror=True              # borders left/right
    ),
)
fig.to_html('Ditribution_counterions_Me_SO4.html')
fig.show()



In [52]:
x = list(energy_to_distance.values())  # mean PF6–cluster distances
y = [(energy - (-66.60981461) - (-0.703061194)*4)* 2625.5 for energy in energy_to_distance.keys()]

fig = px.scatter(
    x=x,
    y=y,
    labels={"x": "Mean SO4-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Energy vs. Mean SO4-Cage Distance",
    width=700,
    height=500
)
fig.update_traces(marker=dict(size=10, color='green', line=dict(width=1, color='black')))  # size=10 for larger points

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()

In [56]:
y_negative = [val for val in y if val < 0]
counts, bins = np.histogram(y_negative, bins=20)
counts_normalized = counts / counts.max() * 100
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig = px.bar(
    x=bin_centers,
    y=counts_normalized,
    labels={"x": "Cohesive energy/ kJ mol\u207B\u00B9", "y": "Relative Counts %"},
    title="Histogram of Relative Energies (Max = 100)",
    color_discrete_sequence=['green']
)

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    bargap=0.05
)

fig.show()

In [59]:
reference_energy = (-66.60981461) + (-0.703061194)*4

distances_all = list(energy_to_distance.values())
energies_all = [(energy - reference_energy) * 2625.5 for energy in energy_to_distance.keys()]

distances = [d for d, e in zip(distances_all, energies_all) if e < 0]
energies = [e for e in energies_all if e < 0]

fig = px.scatter(
    x=distances,
    y=energies,
    labels={"x": "Mean SO4-cage distance/ Å", "y": "Cohesive energy/ kJ mol\u207B\u00B9"},
    title="Relative Energy vs. Mean SO4-cage distance (E < 0)",
    marginal_x="histogram",
    marginal_y="histogram",
    width=800,
    height=600
)

fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=10, color='green', line=dict(width=1, color='black'))
)
fig.update_traces(
    selector=dict(type="histogram"),
    marker=dict(color="green"),
    opacity=0.5
)

for trace in fig.data:
    if trace.type == "histogram":
        trace.nbinsx = 30  # for top histogram (x axis)
        trace.nbinsy = 30  # for right histogram (y axis)
fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", mirror=True, ticks="outside")
)

fig.show()